## Preenfasis

### Configuración

In [ ]:
from pathlib import Path

base_dir = Path(r'C:\Users\RICHI\Documents\Tesis\Data\Andinas3Seg\segmentacion3Seg') # Path of segmented audio files

EXTS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'}

output_root = base_dir.parent / (base_dir.name + '_preenfasis')

# Processing Parameters
alpha = 0.97   # 0.95–0.97 (typical); the higher the alpha, the higher the frequencies
target_rms =     # e.g., 0.1 to set RMS; None to skip
write_outputs = True  # Whether to write processed files to disk

base_dir, output_root

In [ ]:
import numpy as np
import librosa
import soundfile as sf

# Normalization of the peak to [-1, 1]
def peak_normalize(x: np.ndarray, eps: float = 1e-9) -> np.ndarray:
    peak = np.max(np.abs(x)) + eps
    return (x / peak).astype(np.float32)

# Calculating RMS with a small epsilon to avoid division by zero
def rms(x: np.ndarray, eps: float = 1e-12) -> float:
    return float(np.sqrt(np.mean(np.square(x)) + eps))

# RMS normalization to a target value
def rms_normalize(x: np.ndarray, target: float) -> np.ndarray:
    current = rms(x)
    if current == 0:
        return x.astype(np.float32)
    return (x * (target / current)).astype(np.float32)

# Combined Wave Normalization:
def normalize_wave(x: np.ndarray, target_rms: float | None) -> np.ndarray:
    """
    Combined normalization:
      1) Peak to [-1, 1] (ensures range)
      2) Target RMS
      3) Safety clipping to [-1, 1]
    """
    y = peak_normalize(x)
    if target_rms is not None:
        y = rms_normalize(y, target_rms)
        y = np.clip(y, -1.0, 1.0)
    return y.astype(np.float32)

def pre_emphasis(x: np.ndarray, alpha: float = 0.97) -> np.ndarray:
    if x.ndim > 1:
        x = np.mean(x, axis=0)  
    y = np.empty_like(x, dtype=np.float32)
    y[0] = x[0]
    y[1:] = x[1:] - alpha * x[:-1]
    return y

def load_audio(path: Path, sr: int | None = None) -> tuple[np.ndarray, int]:
    x, sr = librosa.load(str(path), sr=sr, mono=True)
    return x.astype(np.float32), sr

def save_audio(path: Path, x: np.ndarray, sr: int):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), x.astype(np.float32), sr)


In [ ]:
from tqdm.auto import tqdm

def process_dataset(input_root: Path, output_root: Path, alpha: float = 0.97, 
                    target_rms: float | None = None, write: bool = True):
    input_root = Path(input_root)
    output_root = Path(output_root)

    files = [p for p in input_root.rglob('*') if p.suffix.lower() in EXTS]
    if not files:
        print('No se encontraron archivos de audio en:', input_root)
        return

    for src in tqdm(files, desc='Procesando audios'):
        rel = src.relative_to(input_root)
        dst = output_root / rel.with_suffix('.wav') 

        try:
            # 1) load
            x, sr = load_audio(src)
            # 2) Normalization (RMS optional) to [-1,1]
            x_norm = normalize_wave(x, target_rms=target_rms)
            # 3) Pre-emphasis
            x_pre = pre_emphasis(x_norm, alpha=alpha)
            # 4) Re-normalize the peak to prevent clipping after the filter
            x_out = peak_normalize(x_pre)

            if write:
                save_audio(dst, x_out, sr)
        except Exception as e:
            print(f'Error en {src}: {e}')

# Run the entire pipeline
process_dataset(base_dir, output_root, alpha=alpha, target_rms=target_rms, write=write_outputs)
print('Salida en:', output_root)


In [ ]:
# ====================== Preview sample (optional) ======================
import random
import matplotlib.pyplot as plt
from IPython.display import Audio, display

candidates = [p for p in base_dir.rglob('*') if p.suffix.lower() in EXTS]
if candidates:
    sample_path = random.choice(candidates)
    x, sr = load_audio(sample_path)
    x_n = normalize_wave(x, target_rms=target_rms)
    x_p = pre_emphasis(x_n, alpha=alpha)
    x_p = peak_normalize(x_p)  # for listening without clipping

    # graph of the first 2 seconds
    n = min(len(x), sr * 2)
    t = np.arange(n) / sr
    plt.figure()
    plt.plot(t, x[:n], label='Original')
    plt.plot(t, x_p[:n], label='Con pre-énfasis')
    plt.title('Señal (primeros 2 s)')
    plt.xlabel('Tiempo [s]')
    plt.ylabel('Amplitud')
    plt.legend()
    plt.show()

    print('Archivo de muestra:', sample_path.name)
    print('Escuchar original:')
    display(Audio(x, rate=sr))
    print('Escuchar con pre-énfasis:')
    display(Audio(x_p, rate=sr))
else:
    print('No hay archivos para vista previa.')

## Data Split

In [ ]:
from pathlib import Path
import random, shutil, csv
from typing import Dict, Tuple, List

IN_ROOT  = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas3Seg\segmentacion3Seg")  # Path of segmented audio files
OUT_ROOT = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas3Seg\Andinas3Seg_Split")  # Path for split audio files
RATIOS   = {"train": 0.70, "val": 0.20, "test": 0.10}
SEED     = 42
AUDIO_EXTS = {".wav", ".flac", ".mp3"} 
OVERWRITE = False 

random.seed(SEED)

def list_tracks_by_genre(in_root: Path) -> Dict[str, List[Path]]:
    """
    Returns a path: genre -> list of ‘TrackFolder’ folders
    Each ‘TrackFolder’ contains the .wav segments for that track.
    """
    mapping = {}
    for genre_dir in sorted(p for p in in_root.iterdir() if p.is_dir()):
        genre = genre_dir.name
        tracks = [p for p in genre_dir.iterdir() if p.is_dir()]
        tracks = [t for t in tracks if any(f.suffix.lower() in AUDIO_EXTS for f in t.iterdir())]
        if tracks:
            mapping[genre] = tracks
    return mapping

def split_indices(n: int, ratios: Dict[str, float]) -> Dict[str, List[int]]:
    idxs = list(range(n))
    random.shuffle(idxs)
    n_train = int(round(ratios["train"] * n))
    n_val   = int(round(ratios["val"]   * n))
    n_test  = max(0, n - n_train - n_val)
    return {
        "train": idxs[:n_train],
        "val":   idxs[n_train:n_train+n_val],
        "test":  idxs[n_train+n_val:]
    }

def copy_track_folder(track_dir: Path, dst_root: Path, genre: str, split: str, overwrite=False):
    dst = dst_root / split / genre / track_dir.name
    dst.mkdir(parents=True, exist_ok=True)
    for f in track_dir.iterdir():
        if not f.is_file() or f.suffix.lower() not in AUDIO_EXTS:
            continue
        dfile = dst / f.name
        if dfile.exists() and not overwrite:
            continue
        shutil.copy2(f, dfile)

def write_manifest(rows: List[Tuple[str,str,str,str]], out_csv: Path):
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with out_csv.open("w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(["split","genre","track_folder","rel_path"])
        w.writerows(rows)

def make_track_split(in_root: Path, out_root: Path, ratios: Dict[str,float], overwrite=False):
    genres_tracks = list_tracks_by_genre(in_root)
    if not genres_tracks:
        raise RuntimeError(f"No se encontraron carpetas de pistas en {in_root}")

    manifest_rows = []
    for genre, tracks in genres_tracks.items():
        if len(tracks) < 3:
            print(f"[AVISO] Género '{genre}' tiene pocas pistas ({len(tracks)}). El split puede quedar desequilibrado.")
        groups = split_indices(len(tracks), ratios)
        for split, idxs in groups.items():
            for i in idxs:
                track_dir = tracks[i]
                copy_track_folder(track_dir, out_root, genre, split, overwrite=overwrite)
                # registra cada archivo copiado en el manifiesto
                dst_base = out_root / split / genre / track_dir.name
                for f in sorted(track_dir.iterdir()):
                    if f.is_file() and f.suffix.lower() in AUDIO_EXTS:
                        rel = (dst_base / f.name).relative_to(out_root).as_posix()
                        manifest_rows.append((split, genre, track_dir.name, rel))

    write_manifest(manifest_rows, out_root / "manifest_tracks_split.csv")
    print("Split por pista completado.")
    print("Salida:", out_root)
    print("Manifiesto:", out_root / "manifest_tracks_split.csv")

if __name__ == "__main__":
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    make_track_split(IN_ROOT, OUT_ROOT, RATIOS, overwrite=OVERWRITE)



## Data Augmentation

### Processing (train only) and saving

In [ ]:
from pathlib import Path
import random, numpy as np
import librosa, soundfile as sf
from tqdm.auto import tqdm

# 1. Settings

TRAIN_ROOT = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas3Seg\Andinas3Seg_Split\train") # Path of train split audio files
AUDIO_EXTS = {".wav"}
SEED = 42

# 2 variations per technique
AUG_PER_TECH = 2

# 3. Ranges
TIME_STRETCH_RANGE = (0.85, 1.15)    
PITCH_SEMITONES_CHOICES = [-4, -3, -2, 2, 3, 4]
SNR_DB_RANGE = (15.0, 25.0)            

TARGET_RMS = 0.1                     

random.seed(SEED)
np.random.seed(SEED)

def list_track_folders(train_root: Path):
    pairs = []
    for gdir in sorted(p for p in train_root.iterdir() if p.is_dir()):
        for tdir in sorted(p for p in gdir.iterdir() if p.is_dir()):
            if any(f.is_file() and f.suffix.lower() in AUDIO_EXTS for f in tdir.iterdir()):
                pairs.append((gdir, tdir))
    return pairs

def rms(x): return float(np.sqrt(np.mean(x*x) + 1e-12))

def match_length(x: np.ndarray, target_len: int):
    if len(x) == target_len: return x
    if len(x) > target_len:  return x[:target_len]
    pad = target_len - len(x)
    L = pad // 2; R = pad - L
    return np.pad(x, (L, R), mode="reflect")

def normalize_to_rms(x: np.ndarray, target_rms: float):
    cur = rms(x)
    if cur < 1e-8: return x
    y = x * (target_rms / cur)
    peak = np.max(np.abs(y))
    if peak > 1.0: y = y / peak
    return y.astype(np.float32)

def load_mono(path: Path):
    y, sr = librosa.load(str(path), sr=None, mono=True)
    return y.astype(np.float32), sr

def save_wav(path: Path, y: np.ndarray, sr: int):
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), y, sr)

# --- Técnicas individuales ---
def apply_time_stretch(y, sr):
    rate = random.uniform(*TIME_STRETCH_RANGE)
    y2 = librosa.effects.time_stretch(y, rate=rate)
    y2 = match_length(y2, len(y))
    return normalize_to_rms(y2, TARGET_RMS), {"time_stretch": rate}

def apply_pitch_shift(y, sr):
    semit = random.choice(PITCH_SEMITONES_CHOICES)
    y2 = librosa.effects.pitch_shift(y, sr=sr, n_steps=semit)
    return normalize_to_rms(y2, TARGET_RMS), {"pitch_semitones": semit}

def apply_noise(y, sr):
    snr_db = random.uniform(*SNR_DB_RANGE)
    p_signal = np.mean(y**2) + 1e-12
    snr_lin = 10.0**(snr_db/10.0)
    p_noise = p_signal / snr_lin
    noise = np.random.normal(0.0, np.sqrt(p_noise), size=y.shape).astype(np.float32)
    y2 = y + noise
    return normalize_to_rms(y2, TARGET_RMS), {"snr_db": snr_db}

TECHS = {
    "TS": apply_time_stretch,
    "PS": apply_pitch_shift,
    "NS": apply_noise,
}

def process_track_with_tech(genre_dir: Path, track_dir: Path, tech_code: str, variant_idx: int):
    dst_track = genre_dir / f"{track_dir.name}__{tech_code}{variant_idx:02d}"
    dst_track.mkdir(parents=True, exist_ok=True)

    params_used = None

    for src in sorted(f for f in track_dir.iterdir() if f.is_file() and f.suffix.lower() in AUDIO_EXTS):
        try:
            y, sr = load_mono(src)
            y_aug, params = TECHS[tech_code](y, sr)
            if params_used is None: params_used = params
            save_wav(dst_track / src.name, y_aug, sr)
        except Exception as e:
            print(f"[WARN] fallo en {src}: {e}")

    (dst_track / "_aug_params.txt").write_text(
        "\n".join([f"{k}={v}" for k, v in (params_used or {}).items()]),
        encoding="utf-8"
    )

def augment_train_per_tech(train_root: Path, aug_per_tech: int):
    pairs = list_track_folders(train_root)
    if not pairs:
        print("No hay pistas en train/.")
        return
    print(f"Pistas: {len(pairs)} | Técnicas: {list(TECHS.keys())} | Variantes/tech: {aug_per_tech}")
    for genre_dir, track_dir in tqdm(pairs, desc="Tracks"):
        for tech_code in TECHS.keys():
            for k in range(1, aug_per_tech+1):
                process_track_with_tech(genre_dir, track_dir, tech_code, k)

if __name__ == "__main__":
    augment_train_per_tech(TRAIN_ROOT, AUG_PER_TECH)
    print("Augmentación por técnica completada (solo train).")



### Combine original and enlarged images for training

In [ ]:
import shutil

train_plus = train_root.parent / 'train_plus_aug' 
for src in train_root.rglob('*'):
    if src.is_file() and src.suffix.lower() in EXTS:
        rel = src.relative_to(train_root)
        dst = train_plus / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

for src in out_root.rglob('*.wav'):
    rel = src.relative_to(out_root)
    dst = train_plus / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print('Carpeta combinada para entrenamiento:', train_plus)


## Log-Mel2D

### Settings

In [ ]:
# %pip install librosa soundfile tqdm pillow --quiet

from pathlib import Path

SPLIT_DIR = Path(r'C:\Users\RICHI\Documents\Tesis\Data\Andinas3Seg\Andinas3Seg_Split') # Path of split audio files
 
# Which folders to process (in order). Adjust as needed:
input_folders = [
    SPLIT_DIR / 'train',  # If it doesn't exist, ignore it
    SPLIT_DIR / 'val',
    SPLIT_DIR / 'test',
]

OUT_ROOT  = Path(r"C:\Users\RICHI\Documents\Tesis\Data\Andinas3Seg\Andinas3Seg_LogMel")  # Output root directory

EXTS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac'}
SPLITS    = ["train", "val", "test"]   # qué splits procesar
# Save Options
SAVE_NPY = False   # stores normalized matrices (float32) in .npy files
SAVE_PNG = True   # Save 8-bit image for inspection/visual use
PNG_CLIP = 3.0    # z-score clipping [-clip, +clip] before mapping to 0..255


### Functions (Hann STFT → Mel 128 → dB → z-score per band)

In [ ]:
import numpy as np
import librosa, soundfile as sf
from tqdm.auto import tqdm
from PIL import Image

def pick_stft_params(sr: int):
    if sr <= 24000:
        n_fft = 1024
        hop_length = int(sr * 0.023)  # ~23 ms
    elif sr <= 44100:
        n_fft = 2048
        hop_length = int(sr * 0.012)  # ~12 ms (entre 10 y 23)
    else:
        n_fft = 4096
        hop_length = int(sr * 0.010)  # ~10 ms
    hop_length = max(1, hop_length)
    win_length = n_fft                  # ventana Hann de tamaño n_fft
    return n_fft, hop_length, win_length

def wav_to_mel_log2d(path, n_mels=128):
    # 1) Carga mono preservando sr original
    x, sr = librosa.load(str(path), sr=None, mono=True)
    # 2) STFT con ventana Hann
    n_fft, hop_length, win_length = pick_stft_params(sr)
    S = librosa.stft(
        x, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window='hann', center=True
    )
    # 3) Espectrograma de potencia
    S_power = np.abs(S) ** 2
    # 4) Banco Mel (128 bandas) y proyección
    M = librosa.feature.melspectrogram(S=S_power, sr=sr, n_mels=n_mels, power=1.0)  # power ya aplicado
    # 5) Escala log (dB)
    M_db = librosa.power_to_db(M, ref=np.max)  # normaliza relativo al máximo
    # 6) Normalización por banda (eje frecuencia)
    #    z-score por cada banda mel a lo largo del tiempo
    mean = M_db.mean(axis=1, keepdims=True)
    std  = M_db.std(axis=1, keepdims=True) + 1e-8
    M_z = (M_db - mean) / std
    return M_z.astype(np.float32)  # [n_mels, T]



### Save

In [ ]:
def save_mel(out_path_base: Path, M_z: np.ndarray, save_npy=True, save_png=True, clip=3.0):
    out_path_base.parent.mkdir(parents=True, exist_ok=True)
    if save_npy:
        np.save(str(out_path_base) + ".npy", M_z.astype(np.float32))
    if save_png:
        Mc  = np.clip(M_z, -clip, clip)
        M01 = (Mc - (-clip)) / (2 * clip)
        img = (M01 * 255.0).round().astype(np.uint8)
        Image.fromarray(img).save(str(out_path_base) + ".png")


### Batch processing (preserves splits and genres)

In [ ]:
def process_split(split_root: Path, out_root: Path):
    if not split_root.exists():
        print(f"[SKIP] No existe: {split_root}")
        return

    split_name = split_root.name  # 'train' | 'val' | 'test'
    files = [p for p in split_root.rglob("*") if p.suffix.lower() in EXTS]
    if not files:
        print(f"[AVISO] Sin archivos de audio en {split_root}")
        return

    ok = fail = 0
    for src in tqdm(files, desc=f"Mel-log2D: {split_name}"):
        # Regarding the split → <class>/.../file.ext
        rel = src.relative_to(split_root)
        # Build an output while preserving the split and class:
        out_base = (out_root / split_name / rel).with_suffix("")  # Remove extension
        try:
            M = wav_to_mel_log2d(src, n_mels=128)   # Your function returns M_z
            save_mel(out_base, M, save_npy=SAVE_NPY, save_png=SAVE_PNG, clip=PNG_CLIP)
            ok += 1
        except Exception as e:
            print(f"[ERROR] {src} -> {e}")
            fail += 1
    print(f"[{split_name}] ok={ok} | errores={fail} | total={ok+fail}")

for split in SPLITS:
    process_split(SPLIT_DIR / split, OUT_ROOT)

print("Listo. Mel-log2D guardados en:", OUT_ROOT)
